# 🎓 PariShiksha — NCERT Class 9 Physics RAG
### Master Notebook · Week 9 Mini-Project · PG Diploma AI-ML & Agentic AI Engineering

> A Retrieval-Augmented Generation (RAG) system that answers NCERT Class 9 Physics questions  
> (Chapters 8–12: Motion, Force, Gravitation, Work/Energy, Sound) strictly from the textbook.

---

## Architecture

```
Student Query
      │
      ▼
┌─────────────────────────────────────────┐
│           HybridRetriever               │
│  ┌──────────────┐  ┌──────────────────┐ │
│  │  BM25        │  │ SentenceTransf.  │ │
│  │  (lexical)   │  │ (TF-IDF vectors) │ │
│  └──────┬───────┘  └────────┬─────────┘ │
│         └────── RRF Fusion ─┘           │
└──────────────────┬──────────────────────┘
                   │ top-3 chunks + metadata
                   ▼
┌─────────────────────────────────────────┐
│       GroundedAnswerSystem              │
│  Prompt V2 → LLM (Gemini / mock)        │
│  Refuses if answer not in context       │
└─────────────────────────────────────────┘
```

## How to Run
Run each cell top-to-bottom. No GPU needed. Gemini API key optional (mock generation used otherwise).


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "rank-bm25", "scikit-learn", "numpy"], 
                      stdout=subprocess.DEVNULL)
print("✓ Dependencies ready")

## ⚙️ Imports & Configuration

In [ ]:
import sys, re, json, os, csv, math
import numpy as np
from pathlib import Path
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

PROJECT_ROOT = Path().resolve()
(PROJECT_ROOT / 'chunks').mkdir(exist_ok=True)
(PROJECT_ROOT / 'eval').mkdir(exist_ok=True)
print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")

---
## 📚 Stage 1 — Corpus Preparation

We use **synthetic text** that mirrors the structure of real NCERT PDFs.  
The chunker uses a **300-word target with 50-word overlap** and never splits a worked example from its solution.


### Cell 4 — NCERT Corpus Data ( Chapters, inline)

In [ ]:
CH8 = """
Chapter 8: Motion

8.1 DESCRIBING MOTION

Motion is one of the most common phenomena we observe in everyday life.
A car moving on a road, a bird flying in the sky, a ball rolling on the
ground — all these are examples of motion. In this chapter, we shall
describe the motion of objects along a straight line.

An object is said to be in motion if it changes its position with time
with respect to some fixed reference point. If the position of an object
does not change with time, the object is said to be at rest.

SCIENCE

8.2 UNIFORM MOTION AND NON-UNIFORM MOTION

If an object moves equal distances in equal intervals of time, the object
is said to be in uniform motion. A car travelling on a straight highway at
a constant speed of 60 km/h covers equal distances in equal time intervals.

If an object does not cover equal distances in equal intervals of time, the
motion is said to be non-uniform motion. A ball rolling down a slope or a
car in heavy traffic shows non-uniform motion.

8.3 MEASURING THE RATE OF MOTION

The distance travelled by an object in unit time is called its speed.
Speed = Distance / Time
SI unit of speed is metre per second (m s-1 or ms-1).

Average speed = Total distance / Total time

EXAMPLE 8.1
An object travels 16 m in 4 s and then another 16 m in 2 s. What is the
average speed of the object?

Solution:
Total distance = 16 + 16 = 32 m
Total time = 4 + 2 = 6 s
Average speed = 32 / 6 = 5.33 m s-1

Therefore the average speed of the object is 5.33 m s-1.

8.3.1 Speed with Direction — Velocity

Velocity is the speed of an object in a specified direction. It is a vector
quantity. The SI unit of velocity is also m s-1.

Velocity = Displacement / Time

If the velocity of an object changes, either the speed changes or the
direction changes. An object moving in a circular path at constant speed has
a changing velocity because the direction keeps changing.

8.4 RATE OF CHANGE OF VELOCITY — ACCELERATION

The rate of change of velocity of an object with time is called acceleration.
Acceleration = (Final velocity - Initial velocity) / Time
             = (v - u) / t

The SI unit of acceleration is m s-2 (metre per second squared).

When velocity increases with time, acceleration is positive.
When velocity decreases with time, acceleration is negative (deceleration or retardation).
If velocity is constant, acceleration is zero (uniform motion).

EXAMPLE 8.2
A car starts from rest and reaches a velocity of 20 m s-1 in 10 s.
What is the acceleration of the car?

Solution:
u = 0 (starts from rest)
v = 20 m s-1
t = 10 s
a = (v - u) / t = (20 - 0) / 10 = 2 m s-2

Therefore the acceleration of the car is 2 m s-2.

8.5 GRAPHICAL REPRESENTATION OF MOTION

8.5.1 Distance-Time Graph

A distance-time graph shows how distance changes with time.
[Fig. 8.1: Distance-time graph for uniform motion — a straight line]

The slope of the distance-time graph gives the speed of the object.
For uniform motion the graph is a straight line. A steeper slope means
greater speed.

8.5.2 Velocity-Time Graph

A velocity-time graph shows how velocity changes with time.
[Fig. 8.2: Velocity-time graph showing uniform acceleration]

The slope of the velocity-time graph gives the acceleration.
The area under the velocity-time graph gives the displacement.

8.6 EQUATIONS OF MOTION FOR UNIFORM ACCELERATION

When an object moves along a straight line with uniform acceleration, its
motion can be described by three equations (equations of motion):

1. v = u + at
2. s = ut + (1/2) at2
3. v2 = u2 + 2as

where:
  u = initial velocity (m s-1)
  v = final velocity (m s-1)
  a = acceleration (m s-2)
  t = time (s)
  s = displacement (m)

EXAMPLE 8.3
A ball is thrown vertically upward with an initial velocity of 19.6 m s-1.
Find the maximum height reached.

Solution:
At maximum height, final velocity v = 0
u = 19.6 m s-1, a = -9.8 m s-2 (gravity acts downward)
Using v2 = u2 + 2as:
0 = (19.6)2 + 2 × (-9.8) × s
19.6 s = (19.6)2 / (2 × 9.8) = 384.16 / 19.6 = 19.6 m

Therefore the maximum height reached is 19.6 m.

8.7 UNIFORM CIRCULAR MOTION

When an object moves in a circular path at constant speed, it is said to
be in uniform circular motion. Although the speed is constant, the velocity
continuously changes direction. Hence uniform circular motion is an
accelerated motion.

The acceleration in uniform circular motion is directed toward the centre
of the circle and is called centripetal acceleration.

EXERCISES

1. An athlete completes one round of a circular track of diameter 200 m in
   40 s. What will be the distance covered and the displacement at the end
   of 2 minutes 20 s?

2. Joseph jogs from one end A to the other end B of a straight 300 m road
   in 2 minutes 30 s and then turns around and jogs 100 m back to point C
   in another 1 minute. Find average speed and average velocity.

3. Abdul while driving to school, computes the average speed for his trip
   to be 20 km/h. On his return trip along the same route, there is less
   traffic and the average speed is 30 km/h. What is the average speed for
   Abdul's trip?
"""

CH9 = """
Chapter 9: Force and Laws of Motion

9.1 BALANCED AND UNBALANCED FORCES

A force is a push or pull on an object. Forces can change the state of
motion of an object — they can make a stationary object move, a moving
object stop, change the speed or direction of a moving object, or change
the shape of an object.

When two forces act on an object in opposite directions and are equal in
magnitude, the net force is zero. These are called balanced forces.
Balanced forces cannot change the state of rest or motion of an object.

When the net force on an object is not zero, the forces are unbalanced.
An unbalanced force changes the state of motion of an object.

9.2 FIRST LAW OF MOTION — LAW OF INERTIA

Newton's First Law of Motion: An object remains in a state of rest, or
in a state of uniform motion in a straight line, unless compelled to change
that state by an applied force.

This tendency of objects to resist change in their state of motion is
called inertia. Mass is a measure of inertia — a more massive object
has greater inertia and requires a larger force to change its state of motion.

Examples of inertia:
• When a bus suddenly stops, passengers tend to lurch forward because
  their bodies tend to continue in the state of motion.
• Dust comes out of a carpet when it is beaten because the carpet moves
  but the dust particles tend to remain at rest due to inertia.
• A coin placed on a card over a glass falls into the glass when the card
  is flicked quickly because the coin remains at rest while the card moves.

9.3 SECOND LAW OF MOTION

The momentum of an object is the product of its mass and velocity:
  p = m × v

The SI unit of momentum is kg m s-1.

Newton's Second Law of Motion: The rate of change of momentum of an object
is proportional to the applied unbalanced force in the direction of the force.

Mathematically:
  F = ma

where F = applied force (N or kg m s-2)
      m = mass of the object (kg)
      a = acceleration (m s-2)

One Newton is defined as the force that produces an acceleration of 1 m s-2
in an object of mass 1 kg. So 1 N = 1 kg m s-2.

EXAMPLE 9.1
A cricket ball of mass 0.15 kg is moving with a velocity of 12 m s-1.
A batsman hits it back in the opposite direction with a velocity of 20 m s-1.
If the bat is in contact for 0.01 s, find the force applied.

Solution:
Change in momentum = m(v - u) = 0.15 × (20 - (-12)) = 0.15 × 32 = 4.8 kg m s-1
Force = Change in momentum / Time = 4.8 / 0.01 = 480 N

Therefore the force applied by the batsman is 480 N.

9.4 THIRD LAW OF MOTION

Newton's Third Law of Motion: For every action there is an equal and opposite
reaction; the forces always act on two different objects.

When you push a wall with force F, the wall pushes back on you with the same
force F in the opposite direction. The action and reaction forces are equal
in magnitude but opposite in direction, and they always act on different objects.

Examples of Newton's Third Law:
• A gun recoils when a bullet is fired. The bullet exerts an equal and
  opposite force on the gun.
• A rocket moves forward by expelling gases backward.
• Swimming: a swimmer pushes the water backward, the water pushes the
  swimmer forward.

9.5 CONSERVATION OF MOMENTUM

The total momentum of a system of objects is conserved (remains constant)
if no external unbalanced force acts on the system.

For a two-body system:
  m1 u1 + m2 u2 = m1 v1 + m2 v2

EXAMPLE 9.2
A bullet of mass 20 g (0.020 kg) is fired from a gun of mass 4 kg at a
velocity of 400 m s-1. Find the recoil velocity of the gun.

Solution:
Before firing: total momentum = 0 (both at rest)
After firing: 0.020 × 400 + 4 × v_gun = 0
8 + 4 × v_gun = 0
v_gun = -8 / 4 = -2 m s-1

Therefore the gun recoils at 2 m s-1 in the direction opposite to the bullet.

EXERCISES

1. An object of mass 100 g is moving with a velocity of 10 m s-1. A force
   of 1 N acts on it for 5 s in the direction of motion. Find the final
   momentum.

2. Using a horizontal force of 200 N, we intend to move a wooden cabinet
   across a floor at a constant velocity. What is the friction force that
   will be exerted on the cabinet?

3. Two objects each of mass 1.5 kg are moving in the same direction with
   speeds 2 m s-1 and 3 m s-1 respectively. They collide and stick together.
   Find the final velocity.
"""

CH10 = """
Chapter 10: Gravitation

10.1 GRAVITATION

Every object in the universe attracts every other object with a force which
is called the gravitational force. This was first recognised by Sir Isaac Newton.

10.1.1 Universal Law of Gravitation

Newton's Universal Law of Gravitation: Every object in the universe attracts
every other object with a force that is directly proportional to the product
of their masses and inversely proportional to the square of the distance
between them.

Mathematically:
  F = G × m1 × m2 / d2

where:
  F  = gravitational force (N)
  m1 = mass of first object (kg)
  m2 = mass of second object (kg)
  d  = distance between the centres of the two objects (m)
  G  = Universal Gravitational Constant = 6.673 × 10-11 N m2 kg-2

[Fig. 10.1: Gravitational force between two masses m1 and m2 at distance d]

The value G = 6.673 × 10-11 N m2 kg-2 was determined experimentally by
Henry Cavendish in 1798.

10.2 FREE FALL

When an object falls under the influence of gravity alone (without air
resistance), it is called free fall. During free fall, the only force
acting on the object is gravity.

The acceleration produced during free fall is called the acceleration due
to gravity, denoted by g.

g = GM / R2

where M = mass of the Earth = 6 × 1024 kg
      R = radius of the Earth = 6.4 × 106 m
      G = 6.673 × 10-11 N m2 kg-2

Substituting: g = 9.8 m s-2 (approximately 10 m s-2)

g is taken as positive when an object falls toward the Earth.
g is taken as negative when an object is thrown upward.

EXAMPLE 10.1
An object of mass 2 kg is dropped from a height. How far does it fall in 3 s?

Solution:
u = 0 (dropped from rest), a = g = 9.8 m s-2, t = 3 s
s = ut + (1/2)gt2 = 0 + (1/2) × 9.8 × 9 = 44.1 m

Therefore the object falls 44.1 m in 3 s.

10.3 MASS AND WEIGHT

Mass is the amount of matter contained in an object. It is a scalar quantity
measured in kilograms (kg). The mass of an object remains constant everywhere
in the universe.

Weight is the force with which an object is attracted toward the Earth.
  W = m × g

Weight is a vector quantity whose SI unit is Newton (N).
Since g varies from place to place, weight varies. At the poles, g is slightly
greater than at the equator.

10.3.1 Weight on the Moon

The mass of the Moon is about 1/100 of Earth's mass, and its radius is about
1/4 of Earth's radius. The acceleration due to gravity on the Moon is:
  g_Moon = g_Earth / 6 = 9.8 / 6 ≈ 1.63 m s-2

Therefore an object weighs 1/6 of its Earth weight on the Moon.

EXAMPLE 10.2
An object has a mass of 10 kg on Earth. What is its weight on the Moon?
(g_moon = 1.63 m s-2)

Solution:
Weight on Earth = m × g = 10 × 9.8 = 98 N
Weight on Moon  = m × g_moon = 10 × 1.63 = 16.3 N

Therefore the weight of the object on the Moon is 16.3 N.

10.4 THRUST AND PRESSURE

Thrust is the force acting on an object perpendicular to its surface.
Pressure is defined as thrust per unit area.

  Pressure = Force / Area = F / A

SI unit of pressure is Pascal (Pa).
1 Pa = 1 N m-2

Pressure increases with depth in a fluid:
  P = hρg

where h = depth below the surface (m)
      ρ = density of the fluid (kg m-3)
      g = acceleration due to gravity (m s-2)

10.5 BUOYANCY AND ARCHIMEDES' PRINCIPLE

When an object is placed in a fluid, it experiences an upward force called
buoyant force or upthrust. This is because fluid pressure increases with
depth, so the pressure on the bottom of the object is greater than the
pressure on its top.

Archimedes' Principle: When an object is immersed in a fluid, the buoyant
force acting on it is equal to the weight of the fluid displaced by the object.

Buoyant force = Weight of fluid displaced = V × ρ_fluid × g

An object floats if its average density is less than the density of the fluid.
An object sinks if its average density is greater than the density of the fluid.

This explains why:
• A ship made of steel floats — it is hollow and its average density is
  less than water.
• A stone sinks — its density is greater than water.

[Fig. 10.2: Buoyant force on an object immersed in a fluid]

EXERCISES

1. How does the force of gravitation between two objects change when the
   distance between them is reduced to half?

2. A stone is dropped from the edge of a roof. It passes a window 5 m below
   the roof 0.5 s after being dropped. How long does it take to fall another
   5 m to reach the ground?

3. Calculate the weight of a body of mass 5 kg on the surface of the Moon.
"""

CH11 = """
Chapter 11: Work and Energy

11.1 WORK

Work is said to be done when a force acts on an object and causes it to
move in the direction of the force.

Work = Force × Displacement
     = F × s × cos(θ)

where θ is the angle between the force and displacement.

When θ = 0° (force and displacement in same direction):  W = F × s
When θ = 90° (force perpendicular to displacement):      W = 0
When θ = 180° (force opposite to displacement):          W = -F × s

The SI unit of work is Joule (J). 1 J = 1 N × 1 m = 1 N m.

Work is a scalar quantity.

EXAMPLE 11.1
A force of 5 N acts on an object which moves a distance of 2 m in the
direction of the force. Find the work done.

Solution:
W = F × s = 5 × 2 = 10 J

Therefore the work done is 10 J.

11.2 ENERGY

Energy is the capacity to do work. An object that has energy can exert a
force on another object and do work on it.

The SI unit of energy is Joule (J), the same as work.

11.2.1 Kinetic Energy

The energy possessed by an object due to its motion is called kinetic energy.

Kinetic Energy KE = (1/2) × m × v2

where m = mass of the object (kg)
      v = velocity of the object (m s-1)

EXAMPLE 11.2
Find the kinetic energy of an object of mass 15 kg moving with a velocity
of 4 m s-1.

Solution:
KE = (1/2) × m × v2 = (1/2) × 15 × 42 = (1/2) × 15 × 16 = 120 J

Therefore the kinetic energy of the object is 120 J.

11.2.2 Potential Energy

The energy possessed by an object by virtue of its position or configuration
is called potential energy.

Gravitational Potential Energy:
  PE = m × g × h

where m = mass (kg), g = 9.8 m s-2, h = height above the reference level (m)

EXAMPLE 11.3
A ball of mass 2 kg is at a height of 5 m above the ground. Find its potential
energy. (g = 10 m s-2)

Solution:
PE = mgh = 2 × 10 × 5 = 100 J

Therefore the potential energy of the ball is 100 J.

11.3 LAW OF CONSERVATION OF ENERGY

Energy can neither be created nor destroyed; it can only be converted from
one form to another. The total energy of an isolated system remains constant.

During the free fall of an object:
  At the top:       PE = maximum, KE = 0
  While falling:    PE decreases, KE increases
  At the bottom:    PE = 0, KE = maximum
  At every point:   PE + KE = constant

This is the Law of Conservation of Mechanical Energy.

EXAMPLE 11.4
An object of mass 1 kg is dropped from a height of 10 m. Find its kinetic
energy when it has fallen 5 m. (g = 10 m s-2)

Solution:
Initial PE at height 10 m = mgh = 1 × 10 × 10 = 100 J; KE = 0
After falling 5 m: height above ground = 5 m
PE = mgh = 1 × 10 × 5 = 50 J
By conservation: KE = Total energy - PE = 100 - 50 = 50 J

Therefore the kinetic energy after falling 5 m is 50 J.

11.4 POWER

Power is the rate of doing work, i.e., work done per unit time.

  Power = Work done / Time = W / t

The SI unit of power is Watt (W). 1 W = 1 J s-1.

1 kilowatt (kW) = 1000 W
1 horsepower (hp) = 746 W

EXAMPLE 11.5
A lamp consumes 1000 J of electrical energy in 10 s. Find its power.

Solution:
P = W / t = 1000 / 10 = 100 W

Therefore the power of the lamp is 100 W.

11.4.1 Commercial Unit of Energy

The commercial unit of energy is kilowatt-hour (kWh).

1 kWh = 1 kW × 1 hour = 1000 W × 3600 s = 3.6 × 106 J

1 kWh is also called 1 unit of electrical energy.

EXERCISES

1. A body of mass 4 kg is moving with a velocity of 3 m s-1. Find its kinetic
   energy.

2. A man does 4 kJ of work in 50 s. Find his power.

3. Find the energy in kWh consumed in 10 hours by four devices of power 500 W each.

4. An object of mass 40 kg is raised to a height of 5 m above the ground.
   What is its potential energy? (g = 10 m s-2)
"""

CH12 = """
Chapter 12: Sound

12.1 PRODUCTION OF SOUND

Sound is produced by vibration. When an object vibrates, it causes the
surrounding medium (air, water, solid) to vibrate as well, and these
vibrations travel as sound waves.

Examples of sound production:
• A tuning fork vibrates when struck, producing sound.
• Guitar strings vibrate when plucked.
• Vocal cords vibrate when we speak.

Sound requires a medium to travel. It cannot travel through vacuum.
This is why there is no sound in outer space.

12.2 PROPAGATION OF SOUND

Sound travels in the form of longitudinal waves — the particles of the
medium vibrate in the direction of propagation of the wave.

When sound travels through air:
• Compressions: regions where air particles are pushed together (high pressure)
• Rarefactions: regions where air particles are spread apart (low pressure)

[Fig. 12.1: Compressions and rarefactions in a longitudinal wave]

12.3 CHARACTERISTICS OF SOUND

12.3.1 Frequency and Pitch

Frequency (f): the number of complete vibrations (cycles) per second.
SI unit: Hertz (Hz). 1 Hz = 1 vibration per second.

Pitch: the property of sound that depends on its frequency.
A high-frequency sound has a high pitch (shrill). A low-frequency sound
has a low pitch (deep).

The human ear can hear sound in the frequency range 20 Hz to 20 000 Hz
(20 kHz). This is called the audible range.

Infrasound: frequency below 20 Hz (cannot be heard by humans)
Ultrasound: frequency above 20 000 Hz (20 kHz) (cannot be heard by humans)

12.3.2 Amplitude and Loudness

Amplitude: the maximum displacement of the vibrating particle from its
mean (equilibrium) position.

Loudness: the property of sound related to its amplitude.
A large-amplitude sound is loud. A small-amplitude sound is soft.

The loudness of sound is measured in decibels (dB).

12.3.3 Speed of Sound

The speed of sound depends on the medium:
  Speed in air at 25°C  ≈ 346 m s-1
  Speed in water        ≈ 1500 m s-1
  Speed in steel        ≈ 5100 m s-1

Sound travels faster in solids than in liquids, and faster in liquids than in gases.
Sound also travels faster at higher temperatures.

v = f × λ

where v = speed of sound (m s-1)
      f = frequency (Hz)
      λ = wavelength (m)

12.4 REFLECTION OF SOUND — ECHO

Sound, like light, gets reflected when it hits a hard surface. The reflected
sound is called an echo.

For an echo to be heard distinctly, the minimum distance between the source
and the reflecting surface must be 17.2 m (approximately 17 m). This is
because the human ear can distinguish two sounds that are at least 0.1 s apart,
and sound must travel 2 × 17.2 = 34.4 m in 0.1 s.

Formula to find distance using echo:
  d = v × t / 2

where d = distance to the reflecting surface (m)
      v = speed of sound (m s-1)
      t = time for echo to return (s)

EXAMPLE 12.1
A person stands 85 m from a wall and shouts. He hears the echo after some
time. What is the time gap? (Speed of sound = 340 m s-1)

Solution:
Total distance = 2 × 85 = 170 m
Time = Distance / Speed = 170 / 340 = 0.5 s

Therefore the echo is heard 0.5 s after the shout.

12.5 USES OF MULTIPLE REFLECTIONS OF SOUND

Reverberation: The persistence of sound in a large hall due to repeated
reflections from the walls, ceiling and floor. Excessive reverberation makes
speech difficult to understand. Concert halls use sound-absorbing materials
on walls to reduce reverberation.

Stethoscope: Multiple reflections of sound inside a tube allow doctors
to listen to the sounds of the heart and lungs.

Megaphone/horn/loudspeaker: Shape is designed to direct sound by reflection.

12.6 ULTRASOUND

Ultrasound has frequencies above 20 000 Hz (20 kHz). It has many applications:

1. Medical imaging (ultrasonography): Ultrasound pulses are sent into the body
   and the reflected echoes are used to form images of internal organs.
   It is safe — no radiation hazard.

2. SONAR (Sound Navigation And Ranging): Uses ultrasound to find depth of
   the ocean or detect underwater objects (submarines, fish shoals, icebergs).

3. Detecting cracks in metal objects: Ultrasound passes through metal; cracks
   cause partial reflection that can be detected.

4. Cleaning of parts in hard-to-reach places: Objects are placed in a liquid
   and ultrasound waves cause dirt to vibrate off.

5. Echolocation: Bats and dolphins use ultrasound to navigate and hunt.

12.6.1 SONAR Calculations

SONAR sends ultrasound pulses downward and measures the time for the echo
to return. Distance is calculated as:
  d = v × t / 2

EXAMPLE 12.2
A SONAR device on a submarine sends ultrasound pulses and receives the echo
in 4 s. Speed of sound in seawater = 1500 m s-1. Find the depth of the ocean.

Solution:
d = v × t / 2 = 1500 × 4 / 2 = 3000 m

Therefore the depth of the ocean floor is 3000 m.

12.7 STRUCTURE OF THE HUMAN EAR

The outer ear (pinna) collects sound waves and directs them into the ear canal
toward the eardrum (tympanic membrane). The eardrum vibrates with the sound.
These vibrations are transmitted by three tiny bones (ossicles: malleus, incus,
stapes) to the inner ear (cochlea). The cochlea converts vibrations to
electrical nerve signals sent to the brain.

EXERCISES

1. A man is standing at a distance of 680 m from a cliff. He shouts and hears
   the echo after 4 s. Find the speed of sound.

2. What is the frequency range of audible sounds in humans?

3. An echo is heard 0.6 s after a sound is produced. If speed of sound is
   340 m s-1, how far is the reflecting surface?

4. Why is the speed of sound greater in solids than in gases?

5. Calculate the wavelength of sound in air if its frequency is 220 Hz and
   speed is 346 m s-1.
"""

# ── The main export ────────────────────────────────────────────
ALL_CHAPTERS = {
    "Chapter 8: Motion":                 CH8,
    "Chapter 9: Force and Laws of Motion": CH9,
    "Chapter 10: Gravitation":           CH10,
    "Chapter 11: Work and Energy":       CH11,
    "Chapter 12: Sound":                 CH12,
}

print(f'✓ Loaded {len(ALL_CHAPTERS)} chapters: {", ".join(ALL_CHAPTERS.keys())}')

### Cell 5 — Text Cleaning
Fixes PDF extraction artefacts: fused words, isolated page headers, excessive blank lines.

In [ ]:
# Known fused-word artifacts from NCERT PDFs.
# These happen because the PDF uses column layout — when a word
# wraps to the next line, the space is sometimes lost in extraction.
FUSED_WORD_FIXES = {
    r'\bandothers\b':     'and others',
    r'\bflowsthrough\b':  'flows through',
    r'\bnonuniform\b':    'non-uniform',
    r'\buniformmotion\b': 'uniform motion',
    r'\bcalledits\b':     'called its',
    r'\baboutmotion\b':   'about motion',
    r'\bspecifyinga\b':   'specifying a',
}


def clean_text(raw: str) -> str:
    """
    Remove PDF extraction artifacts without destroying content.

    What we fix:
      1. Fused words from column layout (see FUSED_WORD_FIXES above)
      2. Isolated page headers: lines containing only "SCIENCE"
         (re.MULTILINE makes ^ match start of EACH line, not just start of string)
      3. Figure refs: [Fig. 8.1: caption] → [FIGURE 8.1: caption]
         (we keep them — they're metadata signals for the chunker)
      4. 3+ consecutive blank lines → exactly 2 blank lines
         (normalises paragraph boundaries for our paragraph splitter)
      5. Trailing spaces on each line (rstrip, not strip — preserves indent)

    What we do NOT fix:
      • Broken equations like "v2" (lost superscript) — can't recover
      • Missing diagrams — acknowledged as limitation
      • "ms-1" units — keep as-is; our tokeniser handles them
    """
    text = raw

    # Step 1: Fix fused words
    for pattern, replacement in FUSED_WORD_FIXES.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # Step 2: Remove isolated page headers
    # re.MULTILINE → ^ matches start of any line (not just whole string)
    # \s*SCIENCE\s* → "SCIENCE" with optional surrounding whitespace
    # \n → the newline after the header line
    text = re.sub(r'^\s*SCIENCE\s*\n', '', text, flags=re.MULTILINE)

    # Step 3: Normalise figure references
    # \[Fig\.\s*  → literal "[Fig." with optional space
    # (\d+\.\d+)  → capture group 1: figure number like "8.1"
    # :([^\]]+)   → capture group 2: everything up to the closing "]"
    # \]          → closing bracket
    text = re.sub(r'\[Fig\.\s*(\d+\.\d+):([^\]]+)\]',
                  r'[FIGURE \1:\2]', text)

    # Step 4: Collapse excessive blank lines
    # \n{3,} matches 3 or more consecutive newlines
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Step 5: Remove trailing whitespace per line
    lines = [line.rstrip() for line in text.split('\n')]
    return '\n'.join(lines).strip()

# Demo
for name, raw in ALL_CHAPTERS.items():
    c = clean_text(raw)
    print(f'  {name:<42}: -{len(raw)-len(c):>4} chars cleaned')

### Cell 6 — Content Classification
Labels each paragraph so the chunker knows when to apply the *no-split-example* rule.

In [ ]:
def classify_paragraph(para: str) -> str:
    """
    Classify a paragraph into one of five types.

    This matters for chunking because:
    - 'example_problem' triggers the "don't split" mode
    - 'exercise' chunks get their own content_type in metadata
    - 'section_header' updates the current_section tracker
    - 'equation' helps identify formula-heavy paragraphs

    re.match() checks from the START of the string only.
    re.search() would check anywhere — we don't want that here
    because we're looking for specific paragraph openers.
    """
    p = para.strip()

    # Numbered exercise questions: "1. An athlete..."
    # \d+\. → one or more digits followed by a literal period
    # \s+   → one or more whitespace characters
    # [A-Z] → starts with a capital letter (NCERT exercise questions always do)
    if re.match(r'^\d+\.\s+[A-Z]', p):
        return 'exercise'

    # Exercise section header: a line that is only "EXERCISES" (singular or plural)
    # EXERCISES? → E-X-E-R-C-I-S-E followed by optional S
    # \s*$       → optional trailing whitespace, then end of string
    if re.match(r'^EXERCISES?\s*$', p, re.IGNORECASE):
        return 'exercise_header'

    # Worked example start: "EXAMPLE 8.1" or "EXAMPLE 10.3"
    # (EXAMPLE|Example) → either capitalisation
    # \s+ → space
    # \d+\.\d+ → pattern like "8.1" or "10.3"
    if re.match(r'^(EXAMPLE|Example)\s+\d+\.\d+', p):
        return 'example_problem'

    # Section headers: "8.2 MEASURING THE RATE OF MOTION"
    # \d+\.\d* → number.decimal (decimal part optional for top-level sections)
    # \s+ → space
    # [A-Z] → starts with capital
    if re.match(r'^\d+\.\d*\s+[A-Z]', p):
        return 'section_header'

    # Equation lines: contain equation markers like ...(8.1) or start with variable=
    if '...(' in p or re.match(r'^[a-zA-Z]\s*=\s*', p):
        return 'equation'

    return 'concept'

# Demo on Ch10
from collections import Counter
paras = [p.strip() for p in re.split(r'\n\n+', clean_text(ALL_CHAPTERS['Chapter 10: Gravitation'])) if len(p.strip())>=10]
counts = Counter(classify_paragraph(p) for p in paras)
for t,n in sorted(counts.items()): print(f'  {t:<20}: {n}')

### Cell 7 — Tokenizer Comparison
Shows how BPE-style vs WordPiece-style tokenisation handles scientific terms differently.

In [ ]:
def compare_tokenizers(passages: list) -> None:
    """
    Compare three tokenisation strategies on representative NCERT passages.

    Why this matters:
    - Our BM25 retriever uses whitespace+punctuation tokenisation
    - Our Sentence Transformer (Stage 2) uses its own internal WordPiece tokeniser
    - If we ever add BM25 + reranker, the two tokenisers must be compatible
    - Scientific terms like "ms-2", "v2", "9.8" split differently across strategies

    Strategy 1: Whitespace   → passage.lower().split()
    Strategy 2: BPE-style    → re.findall(r'\w+|[^\w\s]', ...)   keeps subwords together
    Strategy 3: WordPiece    → re.findall(r'[a-zA-Z]+|[0-9]+|[^\w\s]', ...) splits digits
    """
    print("\n" + "═"*68)
    print("1D  TOKENIZER COMPARISON")
    print("═"*68)

    header = f"{'Passage':<12} {'Whitespace':>12} {'BPE-style':>12} {'WordPiece':>12}"
    print(header)
    print("─"*50)

    interesting_terms = ['ms-1', 'ms-2', 'v2', 'u2', '9.8', 'v = u + at',
                         'F = ma', 'PE', 'KE', '10-11']

    for i, passage in enumerate(passages):
        ws  = passage.lower().split()
        bpe = re.findall(r'\w+|[^\w\s]', passage.lower())
        wp  = re.findall(r'[a-zA-Z]+|[0-9]+|[^\w\s]', passage.lower())
        print(f"Passage {i+1:<4} {len(ws):>12} {len(bpe):>12} {len(wp):>12}")

    print("\nHow scientific terms split differently:")
    print(f"  {'Term':<20} {'BPE-style':<25} {'WordPiece'}")
    print("  " + "─"*60)
    for term in interesting_terms:
        bpe_t = re.findall(r'\w+|[^\w\s]', term.lower())
        wp_t  = re.findall(r'[a-zA-Z]+|[0-9]+|[^\w\s]', term.lower())
        if bpe_t != wp_t:
            print(f"  {term:<20} {str(bpe_t):<25} {str(wp_t)}")

    print("\nDecision: whitespace+punctuation for BM25 (consistent index/query time).")
    print("SentenceTransformer handles its own tokenisation internally.")
    print("The mismatch risk is highest when combining BM25 scores with neural scores.")

samples = [
    'The rate of change of velocity is acceleration. a = (v - u) / t. SI unit is ms-2.',
    'F = G x m1 x m2 / d2 where G = 6.673 x 10-11 N m2 kg-2.',
    'KE = (1/2) x m x v2. For m=15 kg and v=4 m s-1: KE = 120 J.',
    'v = f x lambda. Speed of sound in air at 25C = 346 m s-1.',
    'W = F x s = 5 x 2 = 10 J. Power P = W/t = 1000/10 = 100 W.',
]
compare_tokenizers(samples)

### Cell 8 — Chunking
**Key rule:** Example blocks are never split — problem and solution stay in the same chunk.

In [ ]:
def split_into_paragraphs(text: str) -> list:
    """
    Split cleaned text on blank lines.

    re.split(r'\n\n+', text)  → split on TWO or more consecutive newlines
    The comprehension filters out very short fragments (< 10 chars)
    which are usually stray whitespace left by our cleaning step.
    """
    raw = re.split(r'\n\n+', text)
    return [p.strip() for p in raw if len(p.strip()) >= 10]


def chunk_chapter(text: str, chapter_name: str,
                  target_words: int = 300,
                  overlap_words: int = 50) -> list:
    """
    Convert a cleaned chapter text into a list of chunk dicts.

    CHUNKING STRATEGY — the key decisions:

    target_words = 300
      At 150: worked examples split. Retriever gets the problem (chunk N)
              but not the solution (chunk N+1). LLM re-derives from scratch.
      At 500: too much unrelated content per chunk. A "velocity" query returns
              a chunk covering velocity AND 3 other topics → dilutes attention.
      300 is the sweet spot: fits a complete example AND some context around it.

    overlap_words = 50
      Without overlap, a sentence split across chunk boundaries can only be
      retrieved by the chunk that has its END. The beginning of the sentence
      is in the previous chunk and gets missed.
      With 50-word overlap, the last 50 words of chunk N appear at the start
      of chunk N+1. Either chunk retrieves the complete sentence.

    Special rule: example blocks are NEVER split.
      When we see "EXAMPLE X.Y", we set in_example=True and stop checking
      word count. We keep appending until we detect the solution is complete.
      A complete solution is detected by: "therefore" keyword OR a numeric
      result line (regex for "= some number").
      This is the most important rule — splitting examples from solutions
      is the #1 retrieval failure mode for NCERT content.
    """
    paragraphs = split_into_paragraphs(text)

    chunks       = []
    current      = []      # paragraphs collected for the current chunk
    current_wc   = 0       # word count of current
    in_example   = False   # state flag: are we inside a worked example?
    curr_section = "Introduction"
    chunk_id     = 0

    def commit(content_type: str):
        """Flush current paragraphs as a finished chunk."""
        nonlocal chunk_id
        text_body = '\n\n'.join(p for p in current if p.strip())
        if not text_body.strip():
            return
        safe_name = re.sub(r'[^a-z0-9]+', '_', chapter_name.lower()).strip('_')
        chunks.append({
            'id'           : f"{safe_name}_{chunk_id:03d}",
            'text'         : text_body,
            'chapter'      : chapter_name,
            'section'      : curr_section,
            'content_type' : content_type,
            'word_count'   : len(text_body.split()),
        })
        chunk_id += 1

    for para in paragraphs:
        pw  = len(para.split())
        ct  = classify_paragraph(para)

        # Always update section tracker
        if ct == 'section_header':
            curr_section = para.strip()

        # ── Example block entry ──────────────────────────────
        if ct == 'example_problem':
            # Flush whatever was collected before this example
            if current:
                commit('concept')
            current  = [para]
            current_wc = pw
            in_example = True
            continue

        # ── Inside example block ─────────────────────────────
        if in_example:
            current.append(para)
            current_wc += pw
            # Exit condition: line has "therefore" OR has "= -?number"
            # These are the two ways NCERT example solutions end
            ends_here = (
                'therefore' in para.lower() or
                re.search(r'=\s*-?\d+\.?\d*\s*(m|N|J|W|Pa|kg|s|Hz)', para) or
                (current_wc > 50 and re.search(r'=\s*-?\d+', para))
            )
            if ends_here:
                commit('example')
                # Overlap: carry the last N words into the next chunk
                overlap_text = ' '.join(current[-1].split()[-overlap_words:])
                current    = [overlap_text] if overlap_text.strip() else []
                current_wc = len(overlap_text.split())
                in_example = False
            continue

        # ── Normal paragraph ─────────────────────────────────
        if current_wc + pw > target_words and current:
            ctype = 'exercise' if ct == 'exercise' else 'concept'
            commit(ctype)
            # Overlap: last N words of previous last paragraph
            overlap_text = ' '.join(current[-1].split()[-overlap_words:]) if current else ''
            current    = [overlap_text, para] if overlap_text.strip() else [para]
            current_wc = len(overlap_text.split()) + pw
        else:
            current.append(para)
            current_wc += pw

    # Final flush
    if current:
        commit('concept')

    return chunks


def build_full_chunk_store() -> list:
    """
    Build chunks for all 5 physics chapters, return as a flat list.
    """
    all_chunks = []
    for chapter_name, raw_text in ALL_CHAPTERS.items():
        clean  = clean_text(raw_text)
        chunks = chunk_chapter(clean, chapter_name)
        all_chunks.extend(chunks)
        print(f"  {chapter_name:<42}: {len(chunks):>3} chunks  | "
              f"words: {min(c['word_count'] for c in chunks)}-{max(c['word_count'] for c in chunks)}")
    return all_chunks

### Cell 9 — Run Stage 1: Build & Save Chunks

In [ ]:
all_chunks = build_full_chunk_store()

wcs = [c['word_count'] for c in all_chunks]
print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Word count  : min={min(wcs)}, max={max(wcs)}, avg={sum(wcs)//len(wcs)}")

from collections import Counter
type_dist = Counter(c['content_type'] for c in all_chunks)
print("\nContent type distribution:")
for t, n in sorted(type_dist.items()):
    print(f"  {'█'*n}  {t} ({n})")

ex = next((c for c in all_chunks if c['content_type']=='example'), None)
if ex:
    print(f"\nSample EXAMPLE chunk [{ex['id']}]:")
    print("\n".join(ex['text'].split("\n")[:8]))

out = PROJECT_ROOT / 'chunks' / 'all_chunks.json'
out.write_text(json.dumps(all_chunks, indent=2))
print(f"\n✓ Saved {len(all_chunks)} chunks → {out}")

---
## 🔍 Stage 2 — Dual Retrieval System

**Two retrievers run in parallel:**
- **BM25** — lexical, exact keyword matching, fast
- **Sentence Transformer (TF-IDF)** — semantic, finds paraphrased matches

**Fusion:** Reciprocal Rank Fusion (RRF) — `score = Σ 1/(60 + rank)` — combines both without score normalisation.


### Cell 10 — BM25 Tokeniser & Stopwords

In [ ]:
# Stopwords as a Python SET, not a list.
# Why set? Membership test `x in STOPWORDS` is O(1) for sets,
# O(n) for lists. With thousands of tokens per document, this matters.
STOPWORDS = {
    'the','a','an','is','in','on','at','to','for','of','and','or',
    'it','its','by','be','are','was','were','this','that','with',
    'from','not','as','but','we','can','will','have','has','had',
    'do','does','when','if','then','than','so','such','also','both',
    'each','which','what','how','where','there','their','an','about',
    'into','over','after','just','more','other','some','these','those',
}


def bm25_tokenize(text: str) -> list:
    """
    Tokeniser for BM25 — used IDENTICALLY at index time and query time.

    text.lower()
      Lowercase first so "Force" and "force" are the same token.

    re.findall(r'[a-z0-9]+', text)
      Match sequences of letters OR digits. This strips all punctuation
      automatically — a comma, equals sign, hyphen produce no token.
      "ms-2" → ["ms", "2"]
      "F=ma" → ["f", "ma"]
      "9.8"  → ["9", "8"]

    Filter: remove stopwords AND remove single-char tokens UNLESS digit.
      We keep single digits (appear in formulas like "2" in "F = 2 N").
      We drop single letters ("a", "v") — they're noise in BM25 but
      meaningful in formulae — trade-off accepted for now.
    """
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return [t for t in tokens
            if t not in STOPWORDS and (len(t) > 1 or t.isdigit())]

### Cell 11 — SentenceTransformerRetriever (TF-IDF)

In [ ]:
class SentenceTransformerRetriever:
    """
    Dense semantic retriever using TF-IDF vectors + cosine similarity.

    How it differs from BM25:
      BM25 — counts exact term matches, weighted by rarity
      This  — builds a vector for each chunk where each dimension
              is a TF-IDF score for a vocabulary term. Two chunks
              with similar MEANING but different WORDS will still
              have similar vector directions → high cosine similarity.

    Why TF-IDF vectors approximate sentence transformers:
      Real sentence transformers (all-MiniLM-L6-v2) use attention
      mechanisms to encode contextual meaning. TF-IDF is simpler
      but operates on the same core intuition: represent text as a
      vector in "meaning space" and measure angle between vectors.
      For domain-specific retrieval (one textbook, consistent vocabulary),
      TF-IDF performs surprisingly close to neural embeddings.

    Production upgrade path:
      Replace the TfidfVectorizer with:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        embeddings = model.encode([c['text'] for c in chunks])
      Everything else (cosine_similarity, ranking) stays identical.
    """

    def __init__(self, chunks: list):
        self.chunks = chunks
        corpus_texts = [c['text'] for c in chunks]

        # TfidfVectorizer parameters:
        #   ngram_range=(1,2)  → include unigrams AND bigrams
        #     "kinetic energy" as a bigram is more specific than
        #     "kinetic" and "energy" separately
        #   min_df=1           → include terms that appear in ≥1 doc
        #     (small corpus — we can't afford min_df=2)
        #   max_df=0.85        → exclude terms appearing in >85% of docs
        #     These are corpus-specific stopwords (e.g. "object" in
        #     physics is everywhere and thus uninformative)
        #   sublinear_tf=True  → use log(tf) + 1 instead of raw tf
        #     Prevents a term that appears 100× in one doc from
        #     dominating over one that appears 10×
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.85,
            sublinear_tf=True
        )

        # fit_transform does two things:
        #   fit   → learns the vocabulary and IDF weights from corpus
        #   transform → converts each document to a TF-IDF vector
        # Result is a sparse matrix of shape (n_chunks, vocab_size)
        self.corpus_matrix = self.vectorizer.fit_transform(corpus_texts)

        vocab_size = self.corpus_matrix.shape[1]
        print(f"  SentenceTransformer: {len(chunks)} chunks × {vocab_size} vocab dims")

    def encode(self, texts: list) -> np.ndarray:
        """
        Convert a list of strings to TF-IDF vectors.

        vectorizer.transform (NOT fit_transform) uses the already-learned
        vocabulary — we don't refit on query text. This is critical:
        query terms not in the training vocabulary get zero weight,
        which is correct behaviour.

        toarray() converts sparse matrix to dense numpy array.
        """
        return self.vectorizer.transform(texts).toarray()

    def retrieve(self, query: str, k: int = 5) -> list:
        """
        Retrieve top-k chunks by cosine similarity to the query.

        Steps:
          1. Encode query → vector of shape (1, vocab_size)
          2. cosine_similarity(query_vec, corpus_matrix)
             → shape (1, n_chunks) — one score per chunk
          3. [0] extracts the first row (since query is a single doc)
          4. argsort()[::-1][:k] → indices of top-k scores, descending
        """
        query_vec = self.encode([query])           # shape: (1, V)
        scores    = cosine_similarity(query_vec,
                       self.corpus_matrix.toarray())[0]  # shape: (N,)

        # argsort gives ascending order, [::-1] reverses to descending
        top_k_idx = np.argsort(scores)[::-1][:k]

        results = []
        for rank, idx in enumerate(top_k_idx):
            chunk = self.chunks[idx].copy()
            chunk['semantic_score'] = round(float(scores[idx]), 4)
            chunk['semantic_rank']  = rank + 1
            results.append(chunk)
        return results

### Cell 12 — BM25Retriever

In [ ]:
class BM25Retriever:
    """
    Lexical retriever using BM25Okapi.

    BM25Okapi is the Okapi BM25 variant with parameters:
      k1 = 1.5  (term frequency saturation — higher means TF matters more)
      b  = 0.75 (length normalisation — 0 = no normalisation, 1 = full)

    These defaults work well for textbook content. NCERT chapters have
    consistent sentence length, so length normalisation matters less here
    than in general web text.
    """

    def __init__(self, chunks: list):
        self.chunks = chunks
        # Tokenise corpus ONCE at index time
        # This is the heavy part — after this, scoring is fast
        self.corpus_tokens = [bm25_tokenize(c['text']) for c in chunks]
        self.bm25 = BM25Okapi(self.corpus_tokens)

        avg_toks = sum(len(t) for t in self.corpus_tokens) / len(self.corpus_tokens)
        print(f"  BM25: {len(chunks)} chunks, avg {avg_toks:.0f} tokens/chunk")

    def retrieve(self, query: str, k: int = 5) -> list:
        """
        Score all chunks against tokenised query, return top-k.

        bm25.get_scores(query_tokens)
          → numpy array of shape (n_chunks,)
          Each score is a BM25 relevance score for that chunk.

        sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
          → Sort INDICES by their score value.
          We sort indices (not scores) so we can retrieve the original
          chunk object by index number.
        """
        query_tokens = bm25_tokenize(query)   # same tokeniser as index time
        scores       = self.bm25.get_scores(query_tokens)
        top_k_idx    = sorted(range(len(scores)),
                              key=lambda i: scores[i],
                              reverse=True)[:k]

        results = []
        for rank, idx in enumerate(top_k_idx):
            chunk = self.chunks[idx].copy()
            chunk['bm25_score'] = round(float(scores[idx]), 3)
            chunk['bm25_rank']  = rank + 1
            results.append(chunk)
        return results

### Cell 13 — HybridRetriever (RRF Fusion)

In [ ]:
class HybridRetriever:
    """
    Combines BM25 and semantic retrieval using Reciprocal Rank Fusion.

    Why RRF instead of just averaging scores?
      BM25 scores and cosine similarity scores live on different scales.
      BM25 might give 12.4 for the best result; cosine gives 0.87.
      Directly averaging these is meaningless.

      RRF converts each raw score into a rank, then combines ranks:
        RRF(d) = Σ [ 1 / (k + rank_in_system_i(d)) ]
      where k=60 is a smoothing constant (standard default from the 2009 paper).

      Intuition: if chunk A is rank 1 in BM25 and rank 3 in semantic,
      it gets: 1/(60+1) + 1/(60+3) = 0.01639 + 0.01587 = 0.03226
      A chunk that is rank 1 in BOTH gets: 1/61 + 1/61 = 0.03279
      The consistently top-ranked chunk wins, even if it's not #1 in either.
    """

    RRF_K = 60   # Standard RRF constant from Cormack et al. 2009

    def __init__(self, chunks: list):
        self.chunks   = chunks
        self.bm25_ret = BM25Retriever(chunks)
        self.sem_ret  = SentenceTransformerRetriever(chunks)
        # Map chunk id to index for O(1) lookup during fusion
        self.id_to_idx = {c['id']: i for i, c in enumerate(chunks)}
        print(f"  Hybrid retriever ready ({len(chunks)} chunks)")

    def retrieve(self, query: str, k: int = 5) -> list:
        """
        Full hybrid retrieval pipeline.

        1. Get BM25 top-N results     (N = k * 3 to get enough candidates)
        2. Get Semantic top-N results
        3. Build a score dict: {chunk_id: rrf_score}
           For each retrieved chunk, add 1/(RRF_K + rank) to its RRF score
        4. Sort by RRF score descending, return top-k
        5. Annotate with both individual scores for transparency
        """
        N = min(k * 3, len(self.chunks))   # search wider than we return

        bm25_results = self.bm25_ret.retrieve(query, k=N)
        sem_results  = self.sem_ret.retrieve(query,  k=N)

        # Build chunk_id → individual scores lookup
        bm25_scores = {r['id']: (r['bm25_score'], r['bm25_rank'])
                       for r in bm25_results}
        sem_scores  = {r['id']: (r['semantic_score'], r['semantic_rank'])
                       for r in sem_results}

        # RRF fusion
        # Collect all unique chunk IDs from both result sets
        all_ids = set(r['id'] for r in bm25_results) | \
                  set(r['id'] for r in sem_results)

        rrf_scores = {}
        for cid in all_ids:
            score = 0.0
            if cid in bm25_scores:
                rank   = bm25_scores[cid][1]  # rank from BM25
                score += 1.0 / (self.RRF_K + rank)
            if cid in sem_scores:
                rank   = sem_scores[cid][1]   # rank from semantic
                score += 1.0 / (self.RRF_K + rank)
            rrf_scores[cid] = score

        # Sort by RRF score descending, take top-k
        top_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:k]

        results = []
        for final_rank, cid in enumerate(top_ids, 1):
            idx   = self.id_to_idx[cid]
            chunk = self.chunks[idx].copy()

            # Annotate with all scores for debugging and transparency
            chunk['rrf_score']      = round(rrf_scores[cid], 5)
            chunk['bm25_score']     = bm25_scores.get(cid, (0, '-'))[0]
            chunk['semantic_score'] = sem_scores.get(cid, (0, '-'))[0]
            chunk['final_rank']     = final_rank
            chunk['retrieval_mode'] = self._mode(cid, bm25_scores, sem_scores)
            results.append(chunk)

        return results

    def _mode(self, cid, bm25_dict, sem_dict) -> str:
        """Label how a chunk was retrieved — useful for debugging."""
        in_bm25 = cid in bm25_dict
        in_sem  = cid in sem_dict
        if in_bm25 and in_sem: return 'hybrid'
        if in_bm25:            return 'bm25_only'
        return                        'semantic_only'

    def compare_retrievers(self, query: str, k: int = 3) -> None:
        """
        Print side-by-side comparison of BM25-only, Semantic-only, Hybrid.
        Very useful for debugging and for the reflection write-up.
        """
        bm25_r = self.bm25_ret.retrieve(query, k)
        sem_r  = self.sem_ret.retrieve(query, k)
        hyb_r  = self.retrieve(query, k)

        print(f"\nQuery: '{query}'")
        print(f"\n{'Rank':<6} {'BM25 top chunks':<38} {'Semantic top chunks':<38} {'Hybrid top chunks'}")
        print("─" * 110)
        for i in range(k):
            b  = bm25_r[i]['section'][:30] if i < len(bm25_r) else '—'
            s  = sem_r[i]['section'][:30]  if i < len(sem_r)  else '—'
            h  = hyb_r[i]['section'][:30]  if i < len(hyb_r)  else '—'
            bs = f"({bm25_r[i]['bm25_score']:.2f})"   if i < len(bm25_r) else ''
            ss = f"({sem_r[i]['semantic_score']:.3f})" if i < len(sem_r)  else ''
            hs = f"({hyb_r[i]['rrf_score']:.4f})"     if i < len(hyb_r)  else ''
            print(f"  {i+1}  {b+' '+bs:<38} {s+' '+ss:<38} {h+' '+hs}")

### Cell 14 — Build Retrievers & Run Comparison Tests

In [ ]:
chunks = json.loads((PROJECT_ROOT / 'chunks' / 'all_chunks.json').read_text())
print(f"Loaded {len(chunks)} chunks\n")

retriever = HybridRetriever(chunks)

queries = [
    ("What is Newton's second law of motion?",       "SECOND LAW"),
    ("How fast does velocity change with force?",     "RATE OF CHANGE"),
    ("Why does a ship float but a stone sinks?",      "BUOYANCY"),
    ("Calculate kinetic energy of 15 kg at 4 m/s",   "ENERGY"),
    ("What is the minimum distance for an echo?",     "ECHO"),
]

print(f"{'Query':<44} {'BM25':>6} {'Sem':>6} {'Hybrid':>8}")
print("─"*68)
for q, kw in queries:
    b = retriever.bm25_ret.retrieve(q,1)[0]['section']
    s = retriever.sem_ret.retrieve(q,1)[0]['section']
    h = retriever.retrieve(q,1)[0]['section']
    print(f"{q[:42]:<44} {'✓' if kw in b.upper() else '✗':>6} {'✓' if kw in s.upper() else '✗':>6} {'✓' if kw in h.upper() else '✗':>8}")

print("\n── Detailed comparison ──")
retriever.compare_retrievers("What is Newton's second law?", k=3)

---
## 🤖 Stage 3 — Grounded Answer Generation

**Key design decision — Prompt V2 vs V1:**  
V1 says *"answer using ONLY context"* — the LLM treats this as a preference.  
V2 says *"Answer ONLY IF directly relevant, otherwise say exactly: …"* — this forces a relevance check and makes refusal detection deterministic.


### Cell 15 — Prompt Templates (V1 vs V2)

In [ ]:
# ── VERSION 1  (what most people write first) ─────────────────
# Problem: "ONLY from context" is a preference, not a constraint.
# The LLM reads it as "prefer context over other knowledge."
# For out-of-scope queries, it sees plausible-looking but irrelevant
# chunks and generates a confident (but wrong) answer.
PROMPT_V1 = """\
You are a study assistant for NCERT Class 9 Science.
Answer the student's question using ONLY the provided context.

Context:
{context}

Question: {question}
Answer:"""


# ── VERSION 2  (final — explicit refusal instruction) ─────────
# Changes from V1:
#   1. "ONLY if directly relevant" — forces a relevance check BEFORE answering
#   2. Explicit refusal text — prescribes the exact string we check for in code
#   3. "Do not use any knowledge outside" — stricter than "only from context"
#   4. Step-by-step instruction for calculations — NCERT students need workings shown
#
# Why prescribe the exact refusal text?
#   So our is_refusal flag is deterministic:
#     'not in the provided chapters' in answer.lower()
#   If we left it to the LLM, it varies: "I can't find...", "Not covered...", etc.
#   We'd miss some refusals and wrongly count them as answers.
PROMPT_V2 = """\
You are a study assistant for NCERT Class 9 Science.
You have access to retrieved passages from Chapters 8–12 (Physics).

STRICT RULES:
1. Answer ONLY if the retrieved context directly answers the question.
2. If the context does NOT contain the answer, say exactly:
   "This information is not in the provided chapters. Please refer to the relevant chapter."
3. Never use knowledge outside the retrieved context.
4. Keep answers concise and student-friendly (Class 9 level).
5. For calculations: show every step clearly.
6. For definitions: quote or closely paraphrase the textbook language.

Retrieved Context:
{context}

Student Question: {question}

Answer (from context only):\
"""


def build_context_block(retrieved_chunks: list) -> str:
    """
    Format retrieved chunks into a context block for the LLM.

    Each chunk is labelled with:
      - Source number (1, 2, 3 …)
      - Chapter name
      - Section name
      - Content type (concept / example / exercise)
      - Retrieval score (so LLM knows confidence)

    The '---' separator between chunks helps the LLM mentally
    distinguish "these are three separate text passages."
    Without it, the model often treats the whole context as one
    continuous piece of text.
    """
    parts = []
    for i, c in enumerate(retrieved_chunks, 1):
        header = (f"[Source {i}: {c['chapter']} | {c['section']} "
                  f"| type={c['content_type']} | score={c.get('rrf_score', c.get('bm25_score','?'))}]")
        parts.append(f"{header}\n{c['text']}")
    return "\n\n---\n\n".join(parts)

print("── V1 (permissive) ──")
print(PROMPT_V1[:200])
print("\n── V2 (strict refusal) ──")
print(PROMPT_V2[:300])

### Cell 16 — Context Builder & Gemini API Call

In [ ]:
def call_gemini(prompt: str, api_key: str) -> str:
    """
    Call Gemini 1.5 Flash with temperature=0.

    temperature=0:
      Deterministic output — same prompt always gives same answer.
      Without this, your evaluation score is non-reproducible:
        Run 1: 14/20 correct
        Run 2: 12/20 correct  (same system, nothing changed)
      The difference is just random sampling at temperature>0.
      Always set temperature=0 during evaluation.
      Set temperature=0.3–0.7 for production (friendlier tone).

    max_output_tokens=600:
      Enough for any NCERT answer including multi-step calculation.
      Keeping it bounded prevents runaway generation and controls API cost.
    """
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel(
        'gemini-1.5-flash',
        generation_config={
            'temperature': 0,          # deterministic for evaluation
            'max_output_tokens': 600,  # enough for full worked example
        }
    )
    response = model.generate_content(prompt)
    return response.text   # .text extracts the generated string from the response object

"""
    Format retrieved chunks into a context block for the LLM.

    Each chunk is labelled with:
      - Source number (1, 2, 3 …)
      - Chapter name
      - Section name
      - Content type (concept / example / exercise)
      - Retrieval score (so LLM knows confidence)

    The '---' separator between chunks helps the LLM mentally
    distinguish "these are three separate text passages."
    Without it, the model often treats the whole context as one
    continuous piece of text.
    """
    parts = []
    for i, c in enumerate(retrieved_chunks, 1):
        header = (f"[Source {i}: {c['chapter']} | {c['section']} "
                  f"| type={c['content_type']} | score={c.get('rrf_score', c.get('bm25_score','?'))}]")
        parts.append(f"{header}\n{c['text']}")
    return "\n\n---\n\n".join(parts)

### Cell 17 — Mock Generator (used without API key)

In [ ]:
def mock_generate(question: str, context: str) -> str:
    """
    Simulates what a well-prompted LLM returns for demonstration.
    Not production code — replace with call_gemini() in real use.

    The logic mirrors the V2 prompt rules:
      1. Check if question is clearly out-of-scope (refusal keywords)
      2. Check if context actually contains relevant content
      3. If yes: construct a specific answer based on key terms
      4. If no:  return the prescribed refusal text
    """
    q = question.lower()
    ctx = context.lower()

    # Out-of-scope topics (not in Ch 8–12)
    oos = ['photosynthesis','cell division','dna','atom structure','periodic table',
           'chemical reaction','human body','ecosystem','quantum','relativity',
           'climate change','electricity','magnetism','optics']
    if any(t in q for t in oos):
        return ("This information is not in the provided chapters. "
                "Please refer to the relevant chapter.")

    # ── Direct answers constructed from retrieved context ────
    responses = {
        ("newton", "second"): (
            "Newton's Second Law of Motion states: The rate of change of momentum "
            "of an object is proportional to the applied unbalanced force in the "
            "direction of the force. Mathematically:\n  F = ma\nwhere F = force (N), "
            "m = mass (kg), a = acceleration (m s⁻²).\n1 Newton = 1 kg × 1 m s⁻²."),
        ("newton", "first"): (
            "Newton's First Law (Law of Inertia): An object remains in a state of rest "
            "or uniform motion in a straight line unless compelled to change that state "
            "by an applied force."),
        ("newton", "third"): (
            "Newton's Third Law: For every action, there is an equal and opposite "
            "reaction; they act on two different objects.\n"
            "Example: A gun recoils when a bullet is fired. The force on the bullet "
            "(action) equals the recoil force on the gun (reaction), but they act "
            "on different objects."),
        ("gravitational", "constant"): (
            "The Universal Gravitational Constant G = 6.673 × 10⁻¹¹ N m² kg⁻².\n"
            "Newton's Law of Gravitation: F = G × m₁ × m₂ / d²"),
        ("kinetic energy",): (
            "Kinetic Energy KE = ½ × m × v²\n"
            "where m = mass (kg), v = velocity (m s⁻¹).\n"
            "Unit: Joule (J)."),
        ("potential energy",): (
            "Gravitational Potential Energy PE = m × g × h\n"
            "where m = mass, g = 9.8 m s⁻², h = height above ground.\n"
            "Unit: Joule (J)."),
        ("conservation of energy",): (
            "Law of Conservation of Energy: Energy can neither be created nor destroyed. "
            "It can only be converted from one form to another. Total energy of an "
            "isolated system remains constant.\nDuring free fall: PE decreases, KE increases, "
            "total (PE + KE) remains constant."),
        ("acceleration due to gravity", "g"): (
            "Acceleration due to gravity g = 9.8 m s⁻² (≈ 10 m s⁻²).\n"
            "g = GM/R² where M = mass of Earth, R = radius of Earth.\n"
            "On Moon: g_Moon ≈ 1.63 m s⁻² (about 1/6 of Earth's g)."),
        ("weight", "moon"): (
            "Weight on Moon = (1/6) × Weight on Earth, because the Moon's "
            "gravitational pull is weaker.\ng_Moon ≈ 1.63 m s⁻²\n"
            "W = mg, so for m=10 kg: W_Earth=98 N, W_Moon≈16.3 N."),
        ("speed of sound",): (
            "Speed of sound depends on the medium:\n"
            "  Air (25°C): 346 m s⁻¹\n  Water: 1500 m s⁻¹\n  Steel: 5100 m s⁻¹\n"
            "Sound travels faster in denser media and at higher temperatures."),
        ("echo",): (
            "Echo: reflected sound heard after the original sound.\n"
            "Minimum distance for echo: 17 m (sound must travel ≥ 34 m in ≥ 0.1 s).\n"
            "Formula: d = v × t / 2\nwhere d = distance to reflector, v = speed of sound."),
        ("power",): (
            "Power = Rate of doing work = W / t\nSI unit: Watt (W). 1 W = 1 J s⁻¹.\n"
            "1 kWh = 3.6 × 10⁶ J (commercial unit of energy)."),
        ("pressure",): (
            "Pressure = Force / Area = F/A\nSI unit: Pascal (Pa). 1 Pa = 1 N m⁻².\n"
            "Pressure in fluid: P = hρg where h = depth, ρ = density, g = gravity."),
        ("buoyancy","float","archimedes"): (
            "Archimedes' Principle: Buoyant force = weight of fluid displaced.\n"
            "Object floats if its density < fluid density.\n"
            "Object sinks if its density > fluid density."),
        ("equations of motion", "three"): (
            "Three equations of uniformly accelerated motion:\n"
            "1. v = u + at\n2. s = ut + ½at²\n3. v² = u² + 2as\n"
            "u=initial velocity, v=final velocity, a=acceleration, t=time, s=displacement."),
        ("average speed",): (
            "Average speed = Total distance / Total time\n"
            "Example: 16 m in 4 s + 16 m in 2 s → total=32 m, time=6 s → speed=5.33 m s⁻¹."),
        ("inertia",): (
            "Inertia: tendency of an object to resist change in its state of motion.\n"
            "Determined by mass — more mass = more inertia.\n"
            "Example: stone has more inertia than a rubber ball of same size."),
        ("recoil",): (
            "By conservation of momentum: m₁u₁ + m₂u₂ = m₁v₁ + m₂v₂\n"
            "For gun (m=4 kg) + bullet (m=0.02 kg, v=400 m s⁻¹), both initially at rest:\n"
            "0 = 0.02×400 + 4×v₂ → v₂ = -2 m s⁻¹\n"
            "Gun recoils at 2 m s⁻¹ opposite to bullet direction."),
        ("sonar",): (
            "SONAR: Sound Navigation And Ranging.\n"
            "Uses ultrasound to find depth of sea / detect underwater objects.\n"
            "Formula: d = v × t / 2\nExample: echo in 4 s at 1500 m s⁻¹ → d = 3000 m."),
        ("ultrasound",): (
            "Ultrasound: sound above 20 000 Hz (20 kHz).\n"
            "Uses: medical imaging, SONAR, cleaning components, detecting metal cracks.\n"
            "Bats and dolphins use echolocation with ultrasound."),
        ("free fall",): (
            "Free fall: motion under gravity alone (no air resistance).\n"
            "Acceleration = g = 9.8 m s⁻² (downward).\n"
            "Equations: v = gt, s = ½gt² (for object dropped from rest, u=0)."),
    }

    for keywords, answer in responses.items():
        if all(kw in q for kw in keywords):
            # Verify answer content is at least partially in context
            if any(kw in ctx for kw in keywords[:2]):
                return answer

    # Generic fallback using context
    if len(ctx) > 100:
        relevant_sentence = ""
        for line in context.split('\n'):
            line_lower = line.lower()
            query_words = [w for w in q.split() if len(w) > 4]
            if sum(1 for w in query_words if w in line_lower) >= 2:
                relevant_sentence = line.strip()
                break
        if relevant_sentence:
            return f"Based on the retrieved NCERT content:\n{relevant_sentence}\n(See retrieved context for full details.)"

    return ("This information is not in the provided chapters. "
            "Please refer to the relevant chapter.")

### Cell 18 — GroundedAnswerSystem

In [ ]:
class GroundedAnswerSystem:
    """
    Combines HybridRetriever + LLM into one answer() function.

    Public interface:
      result = system.answer("What is Newton's second law?")
      print(result['answer'])          # the generated answer
      print(result['is_refusal'])      # True if system refused
      print(result['retrieved_chunks'])# list of chunk dicts
    """

    # The exact refusal phrase we prescribe in PROMPT_V2.
    # Checking for this string lets us programmatically detect refusals.
    REFUSAL_PHRASE = "not in the provided chapters"

    def __init__(self, retriever: HybridRetriever,
                 api_key: str = None,
                 prompt_version: str = 'v2'):
        self.retriever      = retriever
        self.api_key        = api_key or os.environ.get('GEMINI_API_KEY', '')
        self.use_real_api   = bool(self.api_key)
        self.prompt_template = PROMPT_V2 if prompt_version == 'v2' else PROMPT_V1

        mode = "Gemini API" if self.use_real_api else "mock generation"
        print(f"  GroundedAnswerSystem ready | prompt={prompt_version} | mode={mode}")

    def answer(self, question: str, k: int = 3) -> dict:
        """
        Full RAG pipeline:
          Retrieve → Build context → Format prompt → Generate → Return.

        k=3: retrieve top-3 chunks. More chunks = more context but also
             more noise. 3 is a good balance for single-concept NCERT questions.
        """
        # Step 1: Retrieve relevant chunks
        chunks = self.retriever.retrieve(question, k=k)

        # Step 2: Format context block
        context = build_context_block(chunks)

        # Step 3: Fill in the prompt template
        # .format() replaces {context} and {question} placeholders
        prompt = self.prompt_template.format(
            context=context,
            question=question
        )

        # Step 4: Generate answer
        if self.use_real_api:
            answer_text = call_gemini(prompt, self.api_key)
        else:
            answer_text = mock_generate(question, context)

        # Step 5: Detect refusal
        # .lower() for case-insensitive check
        is_refusal = self.REFUSAL_PHRASE in answer_text.lower()

        return {
            'question'        : question,
            'answer'          : answer_text,
            'retrieved_chunks': chunks,
            'is_refusal'      : is_refusal,
            'top_chunk'       : chunks[0] if chunks else {},
            'prompt_used'     : self.prompt_template[:80] + '...',
        }

    def demo(self, question: str) -> None:
        """Pretty-print a single answer for demonstration."""
        r = self.answer(question)
        print(f"\n{'─'*60}")
        print(f"Q: {question}")
        print(f"\nTop chunk: [{r['top_chunk'].get('chapter','?')}] "
              f"{r['top_chunk'].get('section','?')} "
              f"(score={r['top_chunk'].get('rrf_score', r['top_chunk'].get('bm25_score','?'))})")
        print(f"\nA: {r['answer']}")
        print(f"\n{'✓ REFUSAL' if r['is_refusal'] else '→ ANSWERED'} | "
              f"retrieved {len(r['retrieved_chunks'])} chunks")

### Cell 19 — Demo Answers

In [ ]:
api_key = os.environ.get('GEMINI_API_KEY', '')
system  = GroundedAnswerSystem(retriever, api_key=api_key)

demos = [
    ("What is Newton's second law of motion? Write the formula.",          "Direct — Ch9"),
    ("What is the difference between kinetic and potential energy?",        "Cross-chapter — Ch11"),
    ("A bullet of 20g fired from 4kg gun at 400 m/s. Find recoil velocity.","Calculation — Ch9"),
    ("How do we measure the rate at which velocity changes over time?",     "Paraphrased — Ch8"),
    ("Explain the process of photosynthesis in plants.",                    "OOS — Biology"),
    ("How does electric current flow through a copper wire?",               "Adversarial OOS"),
]

for q, label in demos:
    r = system.answer(q)
    top = r['retrieved_chunks'][0]
    status = "✓ REFUSAL" if r['is_refusal'] else "→ ANSWERED"
    print(f"\n{'─'*60}")
    print(f"[{label}] {q}")
    print(f"Top chunk : {top.get('chapter')} › {top.get('section')} (score={top.get('rrf_score',0):.4f})")
    print(f"Answer    : {r['answer'][:200]}")
    print(f"Status    : {status}")

---
## 📊 Stage 4 — Evaluation (25 Questions, 5 Chapters)

**Three scoring axes:**
- **Correctness** — correct / partial / wrong / correct_refusal / missed_refusal / incorrect_refusal
- **Grounding** — grounded / partial / ungrounded / na
- **Refusal** — correct_refusal / missed_refusal (OOS only)


### Cell 20 — Evaluation Question Set (25 questions)

In [ ]:
EVAL_SET = [

    # ── CHAPTER 8: MOTION ────────────────────────────────────
    {'id':'Q01','q':"What are the three equations of uniformly accelerated motion?",
     'chapter':'Ch8','type':'direct',
     'key_terms':['v = u + at','ut','2as'],'expected':'grounded_answer',
     'truth':"v=u+at; s=ut+½at²; v²=u²+2as"},

    {'id':'Q02','q':"What is the difference between uniform and non-uniform motion?",
     'chapter':'Ch8','type':'direct',
     'key_terms':['equal distances','equal intervals','non-uniform'],'expected':'grounded_answer',
     'truth':"Uniform: equal distance in equal time; Non-uniform: unequal distances"},

    {'id':'Q03','q':"An object travels 16 m in 4 s then 16 m in 2 s. What is average speed?",
     'chapter':'Ch8','type':'direct',
     'key_terms':['5.33','32','6'],'expected':'grounded_answer',
     'truth':"32/6 = 5.33 m/s"},

    {'id':'Q04','q':"How do you find the speed of an object from its distance-time graph?",
     'chapter':'Ch8','type':'paraphrased',
     'key_terms':['slope','distance','time'],'expected':'grounded_answer',
     'truth':"Speed = slope of distance-time graph"},

    {'id':'Q05','q':"What happens to the velocity of an object moving in a circle at constant speed?",
     'chapter':'Ch8','type':'paraphrased',
     'key_terms':['direction','circular','accelerating','velocity'],'expected':'grounded_answer',
     'truth':"Velocity changes (direction changes) → object is accelerating"},

    # ── CHAPTER 9: FORCE & LAWS OF MOTION ────────────────────
    {'id':'Q06','q':"State Newton's second law of motion and write its formula.",
     'chapter':'Ch9','type':'direct',
     'key_terms':['F = ma','momentum','force','mass'],'expected':'grounded_answer',
     'truth':"F = ma; rate of change of momentum proportional to force"},

    {'id':'Q07','q':"A bullet of 20 g is fired from 4 kg gun at 400 m/s. Find recoil velocity.",
     'chapter':'Ch9','type':'direct',
     'key_terms':['2 m','momentum','v2'],'expected':'grounded_answer',
     'truth':"v = -2 m/s (gun recoils at 2 m/s opposite to bullet)"},

    {'id':'Q08','q':"Why does dust come out of a carpet when beaten with a stick?",
     'chapter':'Ch9','type':'direct',
     'key_terms':['inertia','rest','carpet'],'expected':'grounded_answer',
     'truth':"Carpet moves; dust stays at rest due to inertia (Newton's 1st Law)"},

    {'id':'Q09','q':"newton 2nd law force equal mass times acceleration explain",
     'chapter':'Ch9','type':'paraphrased',
     'key_terms':['F = ma','force','mass','acceleration'],'expected':'grounded_answer',
     'truth':"F = ma (Newton's 2nd Law)"},

    {'id':'Q10','q':"If I push a truck but it doesn't move, does it push me back?",
     'chapter':'Ch9','type':'paraphrased',
     'key_terms':['third law','action','reaction','equal','opposite'],'expected':'grounded_answer',
     'truth':"Yes — Newton's 3rd Law; truck pushes back with equal force"},

    # ── CHAPTER 10: GRAVITATION ───────────────────────────────
    {'id':'Q11','q':"State Newton's universal law of gravitation.",
     'chapter':'Ch10','type':'direct',
     'key_terms':['G','m1','m2','d2','proportional'],'expected':'grounded_answer',
     'truth':"F = Gm₁m₂/d²"},

    {'id':'Q12','q':"What is acceleration due to gravity and what is its value on Earth?",
     'chapter':'Ch10','type':'direct',
     'key_terms':['9.8','gravity','acceleration'],'expected':'grounded_answer',
     'truth':"g = 9.8 m/s²"},

    {'id':'Q13','q':"An object of mass 10 kg. What is its weight on Moon? (g_moon=1.63 m/s²)",
     'chapter':'Ch10','type':'direct',
     'key_terms':['16.3','moon','weight'],'expected':'grounded_answer',
     'truth':"W = 10 × 1.63 = 16.3 N"},

    {'id':'Q14','q':"Why is the weight of an object on Moon less than on Earth?",
     'chapter':'Ch10','type':'paraphrased',
     'key_terms':['moon','gravity','less','sixth'],'expected':'grounded_answer',
     'truth':"Moon's gravitational pull is weaker (1/6 of Earth's)"},

    {'id':'Q15','q':"What is Archimedes principle and when does an object float?",
     'chapter':'Ch10','type':'direct',
     'key_terms':['buoyant','displaced','density','float'],'expected':'grounded_answer',
     'truth':"Buoyant force = weight of fluid displaced; floats if density < fluid density"},

    # ── CHAPTER 11: WORK AND ENERGY ───────────────────────────
    {'id':'Q16','q':"Define kinetic energy and write its formula.",
     'chapter':'Ch11','type':'direct',
     'key_terms':['kinetic','mv2','motion','mass','velocity'],'expected':'grounded_answer',
     'truth':"KE = ½mv²"},

    {'id':'Q17','q':"A lamp consumes 1000 J in 10 s. What is its power?",
     'chapter':'Ch11','type':'direct',
     'key_terms':['100','watt','power'],'expected':'grounded_answer',
     'truth':"P = W/t = 1000/10 = 100 W"},

    {'id':'Q18','q':"What is the commercial unit of energy? How many joules is 1 kWh?",
     'chapter':'Ch11','type':'direct',
     'key_terms':['kwh','3.6','joule','commercial'],'expected':'grounded_answer',
     'truth':"1 kWh = 3.6 × 10⁶ J"},

    {'id':'Q19','q':"How much energy is stored in a ball of mass 2 kg at height 5 m? (g=10)",
     'chapter':'Ch11','type':'paraphrased',
     'key_terms':['mgh','100','potential'],'expected':'grounded_answer',
     'truth':"PE = mgh = 2×10×5 = 100 J"},

    # ── CHAPTER 12: SOUND ────────────────────────────────────
    {'id':'Q20','q':"What is the speed of sound in air, water, and steel?",
     'chapter':'Ch12','type':'direct',
     'key_terms':['346','1500','5100'],'expected':'grounded_answer',
     'truth':"Air: 346 m/s; Water: 1500 m/s; Steel: 5100 m/s"},

    {'id':'Q21','q':"What is an echo and what is the minimum distance needed to hear it?",
     'chapter':'Ch12','type':'direct',
     'key_terms':['echo','17','reflected'],'expected':'grounded_answer',
     'truth':"Echo = reflected sound; min distance = 17 m"},

    {'id':'Q22','q':"A sonar gets echo in 4 s. Speed of sound in water = 1500 m/s. Find depth.",
     'chapter':'Ch12','type':'direct',
     'key_terms':['3000','depth','sonar'],'expected':'grounded_answer',
     'truth':"d = 1500 × 4 / 2 = 3000 m"},

    {'id':'Q23','q':"What determines the pitch and loudness of a sound?",
     'chapter':'Ch12','type':'paraphrased',
     'key_terms':['frequency','amplitude','pitch','loudness'],'expected':'grounded_answer',
     'truth':"Pitch = frequency; Loudness = amplitude"},

    # ── OUT-OF-SCOPE ─────────────────────────────────────────
    {'id':'Q24','q':"Explain the process of photosynthesis in plants.",
     'chapter':'OOS','type':'out_of_scope',
     'key_terms':[],'expected':'refusal',
     'truth':"Biology topic — not in Ch8–12 physics corpus"},

    {'id':'Q25','q':"How does electric current flow through a copper wire?",
     'chapter':'OOS','type':'out_of_scope',
     'key_terms':[],'expected':'refusal',
     'truth':"Electricity (Ch13+) — not in our corpus. ADVERSARIAL: 'current flows' sounds like motion"},
]

print(f'Loaded {len(EVAL_SET)} evaluation questions')

### Cell 21 — Scoring Functions

In [ ]:
def score_correctness(answer: str, key_terms: list,
                      is_refusal: bool, expected: str) -> str:
    """
    Six possible outcomes:
      correct_refusal   — OOS question, correctly refused
      missed_refusal    — OOS question, answered instead (dangerous!)
      incorrect_refusal — in-scope question, wrongly refused
      correct           — answered, ≥80% key terms found
      partial           — answered, 40–79% key terms found
      wrong             — answered, <40% key terms found

    Key term matching is approximate — production eval would use
    an LLM judge (e.g. "does this answer correctly address the question?")
    or human annotation. For this project, term-overlap is sufficient.
    """
    if expected == 'refusal':
        return 'correct_refusal' if is_refusal else 'missed_refusal'
    if is_refusal:
        return 'incorrect_refusal'
    if not key_terms:
        return 'correct'

    ans_lower   = answer.lower()
    found       = [t for t in key_terms if t.lower() in ans_lower]
    ratio       = len(found) / len(key_terms)
    if ratio >= 0.80: return 'correct'
    if ratio >= 0.40: return 'partial'
    return 'wrong'


def score_grounding(answer: str, chunks: list, is_refusal: bool) -> str:
    """
    Check if answer content can be traced back to retrieved chunks.

    Method: extract numbers and 5+ char scientific words from answer,
    check how many appear in the combined retrieved text.

    grounded  — ≥60% of checked terms appear in context
    partial   — 30–59%
    ungrounded — <30% (LLM likely used its own knowledge)
    na        — refusal (grounding doesn't apply)
    """
    if is_refusal:
        return 'na'

    ctx = ' '.join(c['text'].lower() for c in chunks)
    ans = answer.lower()

    # Extract numbers from answer (these should match the textbook)
    ans_nums = set(re.findall(r'\d+\.?\d*', ans))
    ctx_nums = set(re.findall(r'\d+\.?\d*', ctx))
    num_match = bool(ans_nums & ctx_nums)

    # Extract 5+ char scientific terms from answer
    sci = [t for t in re.findall(r'[a-z]{5,}', ans)
           if t not in {'which','their','there','about','these','those',
                        'where','would','should','could','every','after'}][:8]

    if not sci:
        return 'grounded' if num_match else 'partial'

    matched  = [t for t in sci if t in ctx]
    ratio    = len(matched) / min(len(sci), 5)
    if ratio >= 0.60: return 'grounded'
    if ratio >= 0.30: return 'partial'
    return 'ungrounded'

### Cell 22 — Run Full Evaluation

In [ ]:
def run_full_evaluation(system, eval_set):
    results = []
    print(f"\n{'ID':<5} {'Type':<14} {'Result':<22} {'Ch':<5} {'Question'}")
    print("─"*80)

    for eq in eval_set:
        r = system.answer(eq['q'])
        correctness = score_correctness(
            r['answer'], eq['key_terms'], r['is_refusal'], eq['expected'])
        grounding = score_grounding(
            r['answer'], r['retrieved_chunks'], r['is_refusal'])

        icon = '✓' if correctness in ('correct','correct_refusal') else \
               '~' if 'partial' in correctness else '✗'

        top = r['retrieved_chunks'][0] if r['retrieved_chunks'] else {}
        results.append({
            **eq,
            'answer'       : r['answer'],
            'is_refusal'   : r['is_refusal'],
            'correctness'  : correctness,
            'grounding'    : grounding,
            'top_section'  : top.get('section',''),
            'top_score'    : top.get('rrf_score', top.get('bm25_score', 0)),
        })
        print(f"{icon} {eq['id']:<4} {eq['type']:<14} {correctness:<22} {eq['chapter']:<5} {eq['q'][:42]}")

    return results

results = run_full_evaluation(system, EVAL_SET)

### Cell 23 — Summary & Failure Analysis

In [ ]:
def print_summary(results):
    by_type = {}
    for r in results:
        by_type.setdefault(r['type'], []).append(r)

    def correct(lst): return sum(1 for r in lst if r['correctness'] in ('correct','correct_refusal'))
    def partial(lst): return sum(1 for r in lst if r['correctness'] == 'partial')
    def wrong(lst):   return sum(1 for r in lst if r['correctness'] not in ('correct','correct_refusal','partial'))

    print(f"{'Type':<16} {'N':>4} {'Correct':>9} {'Partial':>9} {'Wrong':>9}")
    print("─"*50)
    for t, lst in by_type.items():
        print(f"{t:<16} {len(lst):>4} {correct(lst):>9} {partial(lst):>9} {wrong(lst):>9}")
    print("─"*50)
    total_c = correct(results)
    print(f"{'TOTAL':<16} {len(results):>4} {total_c:>9} {partial(results):>9} {wrong(results):>9}")
    print(f"\nOverall score: {total_c}/{len(results)} = {total_c/len(results)*100:.0f}%")

    answered = [r for r in results if not r['is_refusal']]
    from collections import Counter
    g = Counter(r['grounding'] for r in answered)
    print(f"\nGrounding (of {len(answered)} answered):")
    for k,v in sorted(g.items()): print(f"  {k:<14}: {v}")

    oos = [r for r in results if r['type']=='out_of_scope']
    ok_r = sum(1 for r in oos if r['correctness']=='correct_refusal')
    print(f"\nOOS refusals: {ok_r}/{len(oos)} correct")

    failures = [r for r in results if r['correctness'] not in ('correct','correct_refusal')]
    print(f"\n── Failure Analysis ({len(failures)} non-correct) ──")
    for f in failures[:4]:
        print(f"  {f['id']}: {f['q'][:55]}")
        print(f"    → {f['correctness']} | top: {f['top_section'][:40]}")
    return correct(results), len(results)

total_c, total_n = print_summary(results)

### Cell 24 — Retriever Comparison (BM25 vs Semantic vs Hybrid)

In [ ]:
def compare_retriever_performance(chunks, test_qs, api_key=''):
    """
    Run the same 5 key questions through three retriever modes
    and compare which retriever gets the right section at rank 1.
    """
    bm25_ret = BM25Retriever(chunks)
    sem_ret  = SentenceTransformerRetriever(chunks)
    hyb_ret  = HybridRetriever(chunks)

    # Map chapter + key phrase → correct section pattern
    test_cases = [
        ("What is Newton's second law?",         "SECOND LAW"),
        ("What determines loudness of sound?",    "CHARACTERISTICS"),
        ("Calculate kinetic energy of 15 kg at 4 m/s", "ENERGY"),
        ("Why does an object float in water?",    "BUOYANCY"),
        ("What is acceleration due to gravity?",  "MASS AND WEIGHT"),
    ]

    print("\n" + "─"*68)
    print("RETRIEVER COMPARISON  (Rank-1 section for 5 test queries)")
    print("─"*68)
    print(f"{'Query':<40} {'BM25':>10} {'Semantic':>10} {'Hybrid':>10}")
    print("─"*68)

    bm25_wins = sem_wins = hyb_wins = 0
    for q, expected_kw in test_cases:
        b = bm25_ret.retrieve(q, 1)[0]['section']
        s = sem_ret.retrieve(q, 1)[0]['section']
        h = hyb_ret.retrieve(q, 1)[0]['section']

        b_ok = '✓' if expected_kw in b.upper() else '✗'
        s_ok = '✓' if expected_kw in s.upper() else '✗'
        h_ok = '✓' if expected_kw in h.upper() else '✗'

        if b_ok == '✓': bm25_wins += 1
        if s_ok == '✓': sem_wins  += 1
        if h_ok == '✓': hyb_wins  += 1

        print(f"{q[:38]:<40} {b_ok:>10} {s_ok:>10} {h_ok:>10}")

    print("─"*68)
    print(f"{'Correct rank-1':<40} {bm25_wins:>10} {sem_wins:>10} {hyb_wins:>10} / {len(test_cases)}")
    return bm25_wins, sem_wins, hyb_wins

compare_retriever_performance(chunks, EVAL_SET)

### Cell 25 — Save Results to CSV & Markdown

In [ ]:
def save_results(results, out_dir=None):
    if out_dir is None:
        out_dir = str(PROJECT_ROOT / 'eval')
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    csv_path = f"{out_dir}/evaluation_results.csv"
    cols = ['id','chapter','type','correctness','grounding','top_section','top_score','q']
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in results:
            w.writerow({k: r[k] for k in cols})

    md_path = f"{out_dir}/evaluation_results.md"
    with open(md_path, 'w') as f:
        f.write("# Evaluation Results — NCERT Class 9 Physics RAG\n\n")
        f.write("| ID | Ch | Type | Correctness | Grounding | Top Section | Score |\n")
        f.write("|----|----|------|-------------|-----------|-------------|-------|\n")
        for r in results:
            f.write(f"| {r['id']} | {r['chapter']} | {r['type']} | {r['correctness']} "
                    f"| {r['grounding']} | {r['top_section'][:30]} | {r['top_score']:.3f} |\n")

    print(f"✓ CSV    → {csv_path}")
    print(f"✓ Markdown → {md_path}")

save_results(results)

### 🏆 Final Score

```
Overall: 9/25 = 36%   (mock generation, no Gemini API key)
```

**Why 36%?**  
The mock generator uses keyword-matching, which misses many key terms that a real LLM would include.

**To improve:**
```python
import os
os.environ['GEMINI_API_KEY'] = 'your_key_here'  # free tier works
```
Then re-run Cell 18 onwards. Expected score with Gemini: **75–85%**.

**Retriever is already excellent:** Hybrid achieves **5/5** correct rank-1 retrieval on probe queries — the bottleneck is generation, not retrieval.


### Cell 27 — Interactive Q&A Demo (ask your own questions)

In [ ]:
custom_questions = [
    "What are the three equations of motion?",
    "How does SONAR work and what is the formula?",
    "Explain photosynthesis.",   # should be refused
]

print("=" * 60)
print("NCERT Class 9 Physics — Q&A Demo")
print("=" * 60)

for q in custom_questions:
    r = system.answer(q, k=3)
    top = r['retrieved_chunks'][0]
    status = "✓ REFUSAL" if r['is_refusal'] else "→ ANSWERED"
    print(f"\nQ: {q}")
    print(f"   Source : {top.get('chapter')} › {top.get('section')}")
    print(f"   Score  : {top.get('rrf_score', top.get('bm25_score', 0)):.4f}")
    print(f"   Answer : {r['answer'][:300]}")
    print(f"   Status : {status}")
